### Read SNOWPACK Output at All Sites and Get the Numbers for Table 2 

created by Cassie Lumbrazo\
last updated: June 2026\
run location: UAS linux\
python environment: **xarray**

In [1]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 
import matplotlib.dates as mdates

sns.set_theme()
plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy
import os

In [2]:
pwd

'/home/cassie/python/repos/snow_modeling_point/sites'

# Open Data and Model Simulations

## Function for Reading SMET Files 

In [3]:
def read_smet(filepath):
    header = {}
    fields = None
    data_start = None

    # Read file and parse header
    with open(filepath, "r") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        line = line.strip()

        # Detect fields line
        if line.startswith("fields"):
            fields = line.split("=")[1].strip().split()

        # Detect start of data
        if line == "[DATA]":
            data_start = i + 1
            break

        # Parse header key-value pairs
        if "=" in line and not line.startswith("["):
            key, value = line.split("=", 1)
            header[key.strip()] = value.strip()

    if fields is None:
        raise ValueError("No 'fields' line found in SMET header.")
    if data_start is None:
        raise ValueError("No [DATA] section found.")

    # Read data into DataFrame
    df = pd.read_csv(
        filepath,
        skiprows=data_start,
        delim_whitespace=True,
        names=fields,
        parse_dates=["timestamp"]
    )

    # Set timestamp as index
    df = df.set_index("timestamp")

    # Convert to xarray
    ds = xr.Dataset.from_dataframe(df)

    return ds, header

### Open SNOWPACK SMet Output

In [4]:
# HRRR-AK Files First 
ds_snowpack_hrrrak_ppsa, header_hrrrak_ppsa = read_smet("/home/cassie/python/models/run_snowpack/sites/ppsa/output/hrrrak_ppsa_WY2020-WY2025_base.smet")
ds_snowpack_hrrrak_tram, header_hrrrak_tram = read_smet("/home/cassie/python/models/run_snowpack/sites/tram/output/hrrrak_tram_WY2020-WY2025_base.smet")
ds_snowpack_hrrrak_heen, header_hrrrak_heen = read_smet("/home/cassie/python/models/run_snowpack/sites/heen/output/hrrrak_heen_WY2020-WY2025_base.smet")

# Met HRRR-AK Files now

ds_snowpack_met_hrrrak_ppsa, header_met_hrrrak_ppsa = read_smet("/home/cassie/python/models/run_snowpack/sites/ppsa/output/met_hrrrak_ppsa_WY2020-WY2025_base.smet") # ppsa has WY2020-WY2025
ds_snowpack_met_hrrrak_tram, header_met_hrrrak_tram = read_smet("/home/cassie/python/models/run_snowpack/sites/tram/output/met_hrrrak_tram_WY2023-WY2025_base.smet") # tram has WY2023-WY2025
ds_snowpack_met_hrrrak_heen, header_met_hrrrak_heen = read_smet("/home/cassie/python/models/run_snowpack/sites/heen/output/met_hrrrak_heen_WY2020-WY2022_base.smet") # heen doesn't have until 2025 

/tmp/ipykernel_1309511/1183788903.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
/tmp/ipykernel_1309511/1183788903.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
/tmp/ipykernel_1309511/1183788903.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
/tmp/ipykernel_1309511/1183788903.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
/tmp/ipykernel_1309511/1183788903.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df 

### Open Observations

In [5]:
# open observations

# HEEN 
file_heen = "/hdd/snow_hydrology/met_station/snotel/heenlatinee/heen_met_2016_2026_cleaned_v1.nc"
ds_obs_heen = xr.open_dataset(file_heen)
ds_obs_heen = ds_obs_heen.sel(time=slice("2019-10-01", "2025-09-30"))

#PPSA 
file_ppsa = "/hdd/snow_hydrology/met_station/ppsa2/pppsa_met_station_data_synoptic_2026-03-20" 
ds_obs_ppsa = xr.open_dataset(file_ppsa)
ds_obs_ppsa = ds_obs_ppsa.sel(time=slice("2019-10-01", "2025-09-30"))


# TRAM 
file_tram = "/hdd/snow_hydrology/met_station/tram/tram_met_station_data_synoptic_2026-03-20"  
ds_obs_tram = xr.open_dataset(file_tram)
ds_obs_tram = ds_obs_tram.sel(time=slice("2019-10-01", "2025-09-30"))

In [6]:
# pick colors for each site and stick to them across all plots
ppsa_color = 'darkviolet'
tram_color = 'maroon'
heen_color = 'teal'

# color hrrr-ak model 
hrrrak_color = 'tab:blue'
met_hrrrak_color = 'tab:green'
obs_color = 'tab:gray'

### Cut to just three years each

In [7]:
ds_obs_heen = ds_obs_heen.sel(time=slice("2019-10-01", "2022-09-30"))
ds_obs_ppsa = ds_obs_ppsa.sel(time=slice("2022-10-01", "2025-09-30"))
ds_obs_tram = ds_obs_tram.sel(time=slice("2022-10-01", "2025-09-30"))

# Open Single .Pro File

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

# =============================================================================
# USER SETTINGS
# =============================================================================

pro_path = Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output/hrrrak_ppsa_WY2020-WY2025_base.pro")
# pro_path = Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output/met_hrrrak_ppsa_WY2020-WY2025_base.pro")

pit_table = pd.DataFrame({
    "pit_name": ["Cold Pit", "Warm Pit"],
    "pit_time": ["2025-02-12 12:00", "2025-04-09 12:00"],
    "obs_HS_cm": [133, 170],
    "obs_bulk_density_kg_m3": [334, 401],
    "obs_mean_temperature_C": [-9.2, 0.0],
    "obs_SWE_mm": [444, 682],
})

# =============================================================================
# PARSE SNOWPACK .PRO FILE
# =============================================================================

def parse_snowpack_pro_profiles(path, codes=("0501", "0502", "0503")):
    """
    Stream-parse SNOWPACK .pro file.

    Codes:
        0501 = element top height above ground (cm)
        0502 = element density (kg m-3)
        0503 = element temperature (deg C)
    """
    profiles = {}
    current_time = None
    current = None
    in_data = False

    def save_profile():
        if current_time is not None and current is not None:
            profiles[current_time] = current

    with Path(path).open("r", errors="replace") as f:
        for line in f:
            line = line.strip()

            if line == "[DATA]":
                in_data = True
                continue

            if not in_data or not line:
                continue

            parts = line.split(",")
            code = parts[0]

            if code == "0500":
                save_profile()
                current_time = pd.to_datetime(parts[1], format="%d.%m.%Y %H:%M:%S")
                current = {}
                continue

            if code in codes and current is not None:
                n = int(float(parts[1]))
                values = np.array([float(x) for x in parts[2:] if x != ""], dtype=float)

                if len(values) != n:
                    print(f"Warning: {current_time} code {code}: expected {n}, got {len(values)}")

                current[code] = values

    save_profile()
    return profiles


# =============================================================================
# CALCULATE BULK PROFILE METRICS
# =============================================================================

def profile_bulk_metrics(profile):
    """
    Calculate snow depth, bulk density, mean temperature, and SWE
    from a single .pro profile.
    """
    heights_cm = np.asarray(profile.get("0501", []), dtype=float)
    density = np.asarray(profile.get("0502", []), dtype=float)
    temperature = np.asarray(profile.get("0503", []), dtype=float)

    if len(heights_cm) == 0 or np.nanmax(heights_cm) <= 0:
        return {
            "model_HS_cm": 0.0,
            "model_bulk_density_kg_m3": np.nan,
            "model_mean_temperature_C": np.nan,
            "model_SWE_mm": 0.0,
            "n_layers": 0,
        }

    valid = (
        np.isfinite(heights_cm)
        & np.isfinite(density)
        & np.isfinite(temperature)
        & (heights_cm > 0)
    )

    heights_cm = heights_cm[valid]
    density = density[valid]
    temperature = temperature[valid]

    # 0501 gives element top heights, so layer thickness is the difference
    # between consecutive element-top heights. First layer starts at 0 cm.
    dz_cm = np.diff(np.r_[0.0, heights_cm])

    good = np.isfinite(dz_cm) & (dz_cm > 0)

    dz_cm = dz_cm[good]
    density = density[good]
    temperature = temperature[good]

    model_HS_cm = np.sum(dz_cm)
    dz_m = dz_cm / 100.0

    # kg m-2 is equivalent to mm water equivalent
    model_SWE_mm = np.sum(density * dz_m)

    model_bulk_density = model_SWE_mm / (model_HS_cm / 100.0)

    # For snow-pit comparison, use layer-thickness-weighted mean temperature
    model_mean_temperature = np.average(temperature, weights=dz_cm)

    return {
        "model_HS_cm": model_HS_cm,
        "model_bulk_density_kg_m3": model_bulk_density,
        "model_mean_temperature_C": model_mean_temperature,
        "model_SWE_mm": model_SWE_mm,
        "n_layers": len(dz_cm),
    }


def get_nearest_profile_metrics(profiles, target_time, tolerance="12h"):
    """
    Extract metrics from the profile nearest to target_time.
    """
    target_time = pd.to_datetime(target_time)
    times = pd.DatetimeIndex(profiles.keys())

    idx = times.get_indexer([target_time], method="nearest")[0]
    profile_time = times[idx]

    offset = abs(profile_time - target_time)

    if offset > pd.Timedelta(tolerance):
        raise ValueError(
            f"Nearest profile to {target_time} is {profile_time}, "
            f"which is outside tolerance {tolerance}"
        )

    metrics = profile_bulk_metrics(profiles[profile_time])
    metrics["profile_time"] = profile_time
    metrics["time_offset_hours"] = offset / pd.Timedelta(hours=1)

    return metrics


# =============================================================================
# RUN FOR PIT DATES
# =============================================================================

profiles = parse_snowpack_pro_profiles(pro_path)

rows = []

for _, pit in pit_table.iterrows():
    model = get_nearest_profile_metrics(
        profiles,
        pit["pit_time"],
        tolerance="24h"
    )

    row = {**pit.to_dict(), **model}
    rows.append(row)

comparison = pd.DataFrame(rows)

# Optional rounding for manuscript table
comparison_rounded = comparison.copy()
round_cols = [
    "model_HS_cm",
    "model_bulk_density_kg_m3",
    "model_mean_temperature_C",
    "model_SWE_mm",
]
comparison_rounded[round_cols] = comparison_rounded[round_cols].round(1)

print(comparison_rounded)

   pit_name          pit_time  obs_HS_cm  obs_bulk_density_kg_m3  \
0  Cold Pit  2025-02-12 12:00        133                     334   
1  Warm Pit  2025-04-09 12:00        170                     401   

   obs_mean_temperature_C  obs_SWE_mm  model_HS_cm  model_bulk_density_kg_m3  \
0                    -9.2         444        102.2                     346.4   
1                     0.0         682        139.2                     459.4   

   model_mean_temperature_C  model_SWE_mm  n_layers        profile_time  \
0                      -6.0         353.9       166 2025-02-12 12:00:00   
1                      -0.0         639.5       231 2025-04-09 12:00:00   

   time_offset_hours  
0                0.0  
1                0.0  


# Print as Table 2

In [19]:
import pandas as pd
import numpy as np

# =============================================================================
# AFTER you create `comparison` from the .pro parser
# =============================================================================

df = comparison.copy()

# =============================================================================
# ADD MODEL-OBSERVATION DIFFERENCE STATISTICS
# =============================================================================

metrics = {
    "HS_cm": ("obs_HS_cm", "model_HS_cm"),
    "Bulk_density_kg_m3": ("obs_bulk_density_kg_m3", "model_bulk_density_kg_m3"),
    "Mean_temperature_C": ("obs_mean_temperature_C", "model_mean_temperature_C"),
    "SWE_mm": ("obs_SWE_mm", "model_SWE_mm"),
}

for metric, (obs_col, mod_col) in metrics.items():
    df[f"{metric}_diff"] = df[mod_col] - df[obs_col]
    df[f"{metric}_abs_error"] = np.abs(df[f"{metric}_diff"])
    df[f"{metric}_pct_error"] = 100 * df[f"{metric}_diff"] / df[obs_col]

# Percent error is not meaningful for temperature when obs = 0 °C
df.loc[df["obs_mean_temperature_C"] == 0, "Mean_temperature_C_pct_error"] = np.nan

# =============================================================================
# BUILD MANUSCRIPT-STYLE MULTI-INDEX TABLE
# =============================================================================

table = pd.DataFrame({
    ("Date", ""): (
        df["pit_name"]
        + "\n"
        + pd.to_datetime(df["pit_time"]).dt.strftime("%-d %b %Y")
    ),

    ("Snow Depth (cm)", "Obs"): df["obs_HS_cm"],
    ("Snow Depth (cm)", "Model"): df["model_HS_cm"],
    ("Snow Depth (cm)", "Model - Obs"): df["HS_cm_diff"],
    ("Snow Depth (cm)", "% Error"): df["HS_cm_pct_error"],

    ("Bulk Snow Density (kg m$^{-3}$)", "Obs"): df["obs_bulk_density_kg_m3"],
    ("Bulk Snow Density (kg m$^{-3}$)", "Model"): df["model_bulk_density_kg_m3"],
    ("Bulk Snow Density (kg m$^{-3}$)", "Model - Obs"): df["Bulk_density_kg_m3_diff"],
    ("Bulk Snow Density (kg m$^{-3}$)", "% Error"): df["Bulk_density_kg_m3_pct_error"],

    ("Mean Temperature (°C)", "Obs"): df["obs_mean_temperature_C"],
    ("Mean Temperature (°C)", "Model"): df["model_mean_temperature_C"],
    ("Mean Temperature (°C)", "Model - Obs"): df["Mean_temperature_C_diff"],

    ("SWE (mm) (kg m$^{-2}$)", "Obs"): df["obs_SWE_mm"],
    ("SWE (mm) (kg m$^{-2}$)", "Model"): df["model_SWE_mm"],
    ("SWE (mm) (kg m$^{-2}$)", "Model - Obs"): df["SWE_mm_diff"],
    ("SWE (mm) (kg m$^{-2}$)", "% Error"): df["SWE_mm_pct_error"],
})

table.columns = pd.MultiIndex.from_tuples(table.columns)

# =============================================================================
# ROUND TABLE FOR MANUSCRIPT READABILITY
# =============================================================================

table_rounded = table.copy()

# Convert numeric columns safely
for col in table_rounded.columns:
    if col[0] != "Date":
        table_rounded[col] = pd.to_numeric(table_rounded[col], errors="coerce")

# Column groups
snow_cols = [col for col in table_rounded.columns if "Snow Depth" in col[0]]
density_cols = [col for col in table_rounded.columns if "Density" in col[0]]
swe_cols = [col for col in table_rounded.columns if "SWE" in col[0]]
temp_cols = [col for col in table_rounded.columns if "Temperature" in col[0]]
pct_cols = [col for col in table_rounded.columns if "% Error" in col[1]]

# Physical quantities rounded to nearest whole number
table_rounded[snow_cols] = table_rounded[snow_cols].round(0)
table_rounded[density_cols] = table_rounded[density_cols].round(0)
table_rounded[swe_cols] = table_rounded[swe_cols].round(0)

# Temperature rounded to one decimal place
table_rounded[temp_cols] = table_rounded[temp_cols].round(1)

# Percent error rounded to one decimal place
table_rounded[pct_cols] = table_rounded[pct_cols].round(1)

# Display whole-number columns as integers, while preserving NaN as <NA>
whole_number_cols = snow_cols + density_cols + swe_cols

for col in whole_number_cols:
    if col not in pct_cols:
        table_rounded[col] = table_rounded[col].astype("Int64")

# =============================================================================
# PRINT TABLE
# =============================================================================

print(table_rounded.to_string(index=False))

                 Date Snow Depth (cm)                           Bulk Snow Density (kg m$^{-3}$)                           Mean Temperature (°C)                   SWE (mm) (kg m$^{-2}$)                          
                                  Obs Model Model - Obs % Error                             Obs Model Model - Obs % Error                   Obs Model Model - Obs                    Obs Model Model - Obs % Error
Cold Pit\n12 Feb 2025             133   102         -31   -23.0                             334   346          12     4.0                  -9.2  -6.0         3.2                    444   354         -90   -20.0
 Warm Pit\n9 Apr 2025             170   139         -31   -18.0                             401   459          58    15.0                   0.0  -0.0        -0.0                    682   640         -42    -6.0


# Now, open two .Pro files and compare those for a table

In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

# =============================================================================
# USER SETTINGS
# =============================================================================

pro_paths = {
    "HRRR-AK Model": Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output/hrrrak_ppsa_WY2020-WY2025_base.pro"),
    "Met Mixed Model": Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output/met_hrrrak_ppsa_WY2020-WY2025_base.pro"),
}

pit_table = pd.DataFrame({
    "pit_name": ["“Cold” Pit", "“Warm” Pit"],
    "pit_time": ["2025-02-12 12:00", "2025-04-09 12:00"],
    "obs_HS_cm": [133, 170],
    "obs_bulk_density_kg_m3": [334, 401],
    "obs_mean_temperature_C": [-9.2, 0.0],
    "obs_SWE_mm": [444, 682],
})

# =============================================================================
# PARSE SNOWPACK .PRO FILE
# =============================================================================

def parse_snowpack_pro_profiles(path, codes=("0501", "0502", "0503")):
    profiles = {}
    current_time = None
    current = None
    in_data = False

    def save_profile():
        if current_time is not None and current is not None:
            profiles[current_time] = current

    with Path(path).open("r", errors="replace") as f:
        for line in f:
            line = line.strip()

            if line == "[DATA]":
                in_data = True
                continue

            if not in_data or not line:
                continue

            parts = line.split(",")
            code = parts[0]

            if code == "0500":
                save_profile()
                current_time = pd.to_datetime(parts[1], format="%d.%m.%Y %H:%M:%S")
                current = {}
                continue

            if code in codes and current is not None:
                n = int(float(parts[1]))
                values = np.array([float(x) for x in parts[2:] if x != ""], dtype=float)
                current[code] = values

    save_profile()
    return profiles


def profile_bulk_metrics(profile):
    heights_cm = np.asarray(profile.get("0501", []), dtype=float)
    density = np.asarray(profile.get("0502", []), dtype=float)
    temperature = np.asarray(profile.get("0503", []), dtype=float)

    if len(heights_cm) == 0 or np.nanmax(heights_cm) <= 0:
        return {
            "HS_cm": np.nan,
            "bulk_density_kg_m3": np.nan,
            "mean_temperature_C": np.nan,
            "SWE_mm": np.nan,
        }

    valid = (
        np.isfinite(heights_cm)
        & np.isfinite(density)
        & np.isfinite(temperature)
        & (heights_cm > 0)
    )

    heights_cm = heights_cm[valid]
    density = density[valid]
    temperature = temperature[valid]

    dz_cm = np.diff(np.r_[0.0, heights_cm])
    good = np.isfinite(dz_cm) & (dz_cm > 0)

    dz_cm = dz_cm[good]
    density = density[good]
    temperature = temperature[good]

    dz_m = dz_cm / 100.0

    HS_cm = np.sum(dz_cm)
    SWE_mm = np.sum(density * dz_m)
    bulk_density = SWE_mm / (HS_cm / 100.0)
    mean_temperature = np.average(temperature, weights=dz_cm)

    return {
        "HS_cm": HS_cm,
        "bulk_density_kg_m3": bulk_density,
        "mean_temperature_C": mean_temperature,
        "SWE_mm": SWE_mm,
    }


def get_nearest_profile_metrics(profiles, target_time, tolerance="24h"):
    target_time = pd.to_datetime(target_time)
    times = pd.DatetimeIndex(profiles.keys())

    idx = times.get_indexer([target_time], method="nearest")[0]
    profile_time = times[idx]
    offset = abs(profile_time - target_time)

    if offset > pd.Timedelta(tolerance):
        raise ValueError(
            f"Nearest profile to {target_time} is {profile_time}, "
            f"outside tolerance {tolerance}"
        )

    metrics = profile_bulk_metrics(profiles[profile_time])
    metrics["profile_time"] = profile_time
    return metrics


# =============================================================================
# EXTRACT METRICS FROM BOTH MODEL FILES
# =============================================================================

rows = []

for _, pit in pit_table.iterrows():
    row = {
        "pit_name": pit["pit_name"],
        "pit_time": pd.to_datetime(pit["pit_time"]),
        "obs_HS_cm": pit["obs_HS_cm"],
        "obs_bulk_density_kg_m3": pit["obs_bulk_density_kg_m3"],
        "obs_mean_temperature_C": pit["obs_mean_temperature_C"],
        "obs_SWE_mm": pit["obs_SWE_mm"],
    }

    for model_name, pro_path in pro_paths.items():
        profiles = parse_snowpack_pro_profiles(pro_path)
        metrics = get_nearest_profile_metrics(
            profiles,
            pit["pit_time"],
            tolerance="24h"
        )

        prefix = (
            model_name
            .replace(" ", "_")
            .replace("-", "")
            .replace("/", "_")
        )

        row[f"{prefix}_HS_cm"] = metrics["HS_cm"]
        row[f"{prefix}_bulk_density_kg_m3"] = metrics["bulk_density_kg_m3"]
        row[f"{prefix}_mean_temperature_C"] = metrics["mean_temperature_C"]
        row[f"{prefix}_SWE_mm"] = metrics["SWE_mm"]
        row[f"{prefix}_profile_time"] = metrics["profile_time"]

    rows.append(row)

comparison = pd.DataFrame(rows)

# =============================================================================
# BUILD TABLE LIKE SCREENSHOT
# =============================================================================

model_cols = {
    "HRRR-AK Model": "HRRRAK_Model",
    "Met Mixed Model": "Met_Mixed_Model",
}

table_rows = []

for _, row in comparison.iterrows():

    # Main data row
    table_rows.append({
        ("Date", ""): row["pit_name"] + "\n" + row["pit_time"].strftime("%-d %b\n%Y"),

        ("Snow Depth\n(cm)", "Obs"): row["obs_HS_cm"],
        ("Snow Depth\n(cm)", "HRRR-AK\nModel"): row["HRRRAK_Model_HS_cm"],
        ("Snow Depth\n(cm)", "Met\nMixed\nModel"): row["Met_Mixed_Model_HS_cm"],

        ("Bulk Snow Density\n(kg m$^{-3}$)", "Obs"): row["obs_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "HRRR-AK\nModel"): row["HRRRAK_Model_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "Met\nMixed\nModel"): row["Met_Mixed_Model_bulk_density_kg_m3"],

        ("Mean Temperature\n(°C)", "Obs"): row["obs_mean_temperature_C"],
        ("Mean Temperature\n(°C)", "HRRR-AK\nModel"): row["HRRRAK_Model_mean_temperature_C"],
        ("Mean Temperature\n(°C)", "Met\nMixed\nModel"): row["Met_Mixed_Model_mean_temperature_C"],

        ("SWE\n(mm)(kg m$^{-2}$)", "Obs"): row["obs_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "HRRR-AK\nModel"): row["HRRRAK_Model_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "Met\nMixed\nModel"): row["Met_Mixed_Model_SWE_mm"],
    })

    # Error row
    table_rows.append({
        ("Date", ""): "Model -\nObs\nError (%)",

        ("Snow Depth\n(cm)", "Obs"): "",
        ("Snow Depth\n(cm)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_HS_cm"] - row["obs_HS_cm"]) / row["obs_HS_cm"],
        ("Snow Depth\n(cm)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_HS_cm"] - row["obs_HS_cm"]) / row["obs_HS_cm"],

        ("Bulk Snow Density\n(kg m$^{-3}$)", "Obs"): "",
        ("Bulk Snow Density\n(kg m$^{-3}$)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_bulk_density_kg_m3"] - row["obs_bulk_density_kg_m3"]) / row["obs_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_bulk_density_kg_m3"] - row["obs_bulk_density_kg_m3"]) / row["obs_bulk_density_kg_m3"],

        ("Mean Temperature\n(°C)", "Obs"): "",
        ("Mean Temperature\n(°C)", "HRRR-AK\nModel"): "",
        ("Mean Temperature\n(°C)", "Met\nMixed\nModel"): "",

        ("SWE\n(mm)(kg m$^{-2}$)", "Obs"): "",
        ("SWE\n(mm)(kg m$^{-2}$)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_SWE_mm"] - row["obs_SWE_mm"]) / row["obs_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_SWE_mm"] - row["obs_SWE_mm"]) / row["obs_SWE_mm"],
    })

table = pd.DataFrame(table_rows)
table.columns = pd.MultiIndex.from_tuples(table.columns)

# =============================================================================
# ROUNDING
# =============================================================================

table_display = table.copy()

snow_cols = [col for col in table_display.columns if "Snow Depth" in col[0]]
density_cols = [col for col in table_display.columns if "Density" in col[0]]
swe_cols = [col for col in table_display.columns if "SWE" in col[0]]
temp_cols = [col for col in table_display.columns if "Temperature" in col[0]]

error_rows = table_display[("Date", "")].astype(str).str.contains("Error", na=False)

def format_value(x, decimals=0):
    if pd.isna(x) or x == "":
        return ""
    return f"{float(x):.{decimals}f}"

# Whole numbers for snow depth, density, and SWE data rows
for col in snow_cols + density_cols + swe_cols:
    table_display[col] = table_display[col].apply(lambda x: format_value(x, decimals=0))

# One decimal for temperature data rows
for col in temp_cols:
    table_display[col] = table_display[col].apply(lambda x: format_value(x, decimals=1))

# Percent error rows: one decimal, but leave Obs columns blank
for col in snow_cols + density_cols + swe_cols:
    if col[1] == "Obs":
        table_display.loc[error_rows, col] = ""
    else:
        table_display.loc[error_rows, col] = table.loc[error_rows, col].apply(
            lambda x: format_value(x, decimals=1)
        )

# Leave temperature error row blank
for col in temp_cols:
    table_display.loc[error_rows, col] = ""

print(table_display.to_string(index=False))

                    Date Snow Depth\n(cm)                                  Bulk Snow Density\n(kg m$^{-3}$)                                  Mean Temperature\n(°C)                                  SWE\n(mm)(kg m$^{-2}$)                                 
                                      Obs HRRR-AK\nModel Met\nMixed\nModel                              Obs HRRR-AK\nModel Met\nMixed\nModel                    Obs HRRR-AK\nModel Met\nMixed\nModel                    Obs HRRR-AK\nModel Met\nMixed\nModel
“Cold” Pit\n12 Feb\n2025              133            102               147                              334            346               378                   -9.2           -6.0              -5.1                    444            354               556
 Model -\nObs\nError (%)                           -23.2              10.6                                             3.7              13.1                                                                                         -20.3       

In [22]:
-9.2 - (-6.0)

-3.1999999999999993

In [23]:
-9.2 - (-5.1)

-4.1